# Module 9 • Machine Translation

# Lesson 56 • Machine Translation End-to-End Project — Data Preparation, Fine-Tuning, Evaluation, and Experiment Reporting

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced Capstone  
**Execution target:** CPU-only core; optional pretrained fine-tuning template

---

## Project Goal

This lesson closes Module 9 with a complete machine-translation experiment.

You will build a reproducible pipeline that includes:

- parallel-corpus validation;
- duplicate and overlap checks;
- train/dev/test splitting;
- source/target preprocessing;
- vocabulary and sequence statistics;
- baseline construction;
- tiny Transformer training;
- decoding;
- BLEU-style evaluation;
- chrF-style evaluation;
- bootstrap confidence intervals;
- paired significance analysis;
- linguistic error analysis;
- experiment-table generation;
- publication-style reporting;
- an optional Hugging Face fine-tuning workflow.

The executable project remains small enough to run on CPU.

## Learning Objectives

After completing this capstone, the learner should be able to:

- validate a bilingual parallel corpus;
- create leakage-safe train/dev/test splits;
- inspect length and vocabulary distributions;
- define preprocessing without contaminating the test set;
- train a tiny Transformer MT baseline;
- decode held-out sentences;
- compute multiple MT metrics;
- quantify uncertainty with bootstrap confidence intervals;
- compare systems with paired statistics;
- perform translation error analysis;
- record experiment configuration;
- design a current pretrained Seq2Seq fine-tuning workflow;
- write a concise final experiment report.

## Table of Contents

1. Experimental Question  
2. Reproducibility Setup  
3. Parallel Corpus  
4. Corpus Validation  
5. Duplicate Detection  
6. Train/Dev/Test Split  
7. Leakage Detection  
8. Source and Target Statistics  
9. Tokenization Policy  
10. Arabic/Tashkeel Policy  
11. Vocabulary Construction  
12. OOV Analysis  
13. Length Analysis  
14. Baseline System  
15. Tiny Transformer Dataset  
16. Padding and Masks  
17. Positional Encoding  
18. Transformer Model  
19. Training Objective  
20. Training Loop  
21. Validation Loss  
22. Checkpoint Selection  
23. Greedy Decoding  
24. Test Translation  
25. BLEU-Style Metric  
26. chrF-Style Metric  
27. Exact Match  
28. Sentence-Level Scores  
29. Mean and Standard Deviation  
30. Bootstrap 95% CI  
31. Paired System Comparison  
32. Approximate Randomization  
33. Error Taxonomy  
34. Automatic Error Flags  
35. Manual Error Sheet  
36. Direction-Specific Analysis  
37. Arabic-Specific Analysis  
38. Tashkeel Preservation  
39. Experiment Configuration  
40. Results Table  
41. Ablation Design  
42. Baseline vs Transformer  
43. Pretrained Fine-Tuning Workflow  
44. Current Hugging Face Seq2Seq Template  
45. Evaluation During Fine-Tuning  
46. Checkpointing  
47. Test-Set Discipline  
48. Final Report Structure  
49. Limitations  
50. Reproducibility Checklist  
51. Knowledge Check  
52. Project Exercises  
53. Module 9 Summary

# 1. Experimental Question

A good MT project begins with a precise question.

Example:

> Does a Transformer-based translation model outperform a lexical baseline on a held-out
> French→English test set, and is the improvement statistically reliable?

A research question should specify:

- translation direction;
- training condition;
- test domain;
- comparison system;
- metrics.

In [ ]:
import copy
import math
import platform
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

print("Device:", DEVICE)

# 2. Reproducibility Setup

Fix random seeds and record:

- software versions;
- hardware;
- dataset split;
- preprocessing;
- hyperparameters;
- decoding settings;
- metric settings.

# 3. Parallel Corpus

The project uses a small controlled French→English corpus so all stages can execute
offline.

In [ ]:
parallel_pairs = [
    ("je suis ici", "i am here"),
    ("je suis petit", "i am small"),
    ("je suis grand", "i am big"),
    ("tu es ici", "you are here"),
    ("tu es petit", "you are small"),
    ("tu es grand", "you are big"),
    ("il est ici", "he is here"),
    ("il est petit", "he is small"),
    ("il est grand", "he is big"),
    ("elle est ici", "she is here"),
    ("elle est petite", "she is small"),
    ("elle est grande", "she is big"),
    ("nous sommes ici", "we are here"),
    ("nous sommes petits", "we are small"),
    ("nous sommes grands", "we are big"),
    ("vous etes ici", "you are here"),
    ("vous etes petits", "you are small"),
    ("vous etes grands", "you are big"),
    ("ils sont ici", "they are here"),
    ("ils sont petits", "they are small"),
    ("ils sont grands", "they are big"),
    ("elles sont ici", "they are here"),
    ("elles sont petites", "they are small"),
    ("elles sont grandes", "they are big"),
]

corpus = pd.DataFrame(
    parallel_pairs,
    columns=["source", "target"],
)

corpus.head()

# 4. Corpus Validation

Before splitting, check:

- empty source/target cells;
- misaligned row counts;
- non-string values;
- abnormal whitespace.

In [ ]:
def validate_parallel_corpus(frame):
    issues = []

    if list(frame.columns) != [
        "source",
        "target",
    ]:
        issues.append(
            "Unexpected columns."
        )

    for index, row in (
        frame.iterrows()
    ):
        if (
            not isinstance(
                row["source"],
                str,
            )
            or not isinstance(
                row["target"],
                str,
            )
        ):
            issues.append(
                f"Non-string row: {index}"
            )
            continue

        if (
            not row["source"].strip()
            or not row["target"].strip()
        ):
            issues.append(
                f"Empty pair: {index}"
            )

    return issues

validation_issues = (
    validate_parallel_corpus(
        corpus
    )
)

validation_issues

# 5. Duplicate Detection

Duplicates can distort evaluation and leak examples across splits.

In [ ]:
duplicate_mask = corpus.duplicated(
    subset=[
        "source",
        "target",
    ],
    keep=False,
)

corpus[
    duplicate_mask
]

# 6. Train/Dev/Test Split

The split must happen **before** vocabulary construction and model selection.

Test data should remain untouched until final evaluation.

In [ ]:
rng = np.random.default_rng(
    SEED
)

indices = np.arange(
    len(corpus)
)

rng.shuffle(
    indices
)

train_end = 16
dev_end = 20

train_indices = indices[
    :train_end
]

dev_indices = indices[
    train_end:dev_end
]

test_indices = indices[
    dev_end:
]

train_frame = (
    corpus.iloc[
        train_indices
    ]
    .reset_index(
        drop=True
    )
)

dev_frame = (
    corpus.iloc[
        dev_indices
    ]
    .reset_index(
        drop=True
    )
)

test_frame = (
    corpus.iloc[
        test_indices
    ]
    .reset_index(
        drop=True
    )
)

pd.Series({
    "train": len(train_frame),
    "dev": len(dev_frame),
    "test": len(test_frame),
})

# 7. Leakage Detection

In [ ]:
def pair_set(frame):
    return set(
        zip(
            frame["source"],
            frame["target"],
        )
    )

train_pairs = pair_set(
    train_frame
)

dev_pairs = pair_set(
    dev_frame
)

test_pairs = pair_set(
    test_frame
)

leakage = pd.Series({
    "train_dev_overlap": len(
        train_pairs
        & dev_pairs
    ),
    "train_test_overlap": len(
        train_pairs
        & test_pairs
    ),
    "dev_test_overlap": len(
        dev_pairs
        & test_pairs
    ),
})

leakage

# 8. Source and Target Statistics

In [ ]:
def corpus_statistics(frame):
    source_lengths = frame[
        "source"
    ].str.split().str.len()

    target_lengths = frame[
        "target"
    ].str.split().str.len()

    return pd.Series({
        "pairs": len(frame),
        "mean_source_tokens": float(
            source_lengths.mean()
        ),
        "max_source_tokens": int(
            source_lengths.max()
        ),
        "mean_target_tokens": float(
            target_lengths.mean()
        ),
        "max_target_tokens": int(
            target_lengths.max()
        ),
    })

pd.DataFrame({
    "train": corpus_statistics(
        train_frame
    ),
    "dev": corpus_statistics(
        dev_frame
    ),
    "test": corpus_statistics(
        test_frame
    ),
})

# 9. Tokenization Policy

This project uses whitespace tokenization for transparency.

Real systems usually use subword tokenization.

In [ ]:
def tokenize(text):
    return text.split()

tokenize(
    "je suis ici"
)

# 10. Arabic/Tashkeel Policy

For Arabic↔English experiments, the project specification should explicitly state
whether Arabic is:

- fully vocalized;
- partially vocalized;
- unvocalized.

If the task is fully vocalized, do not strip tashkeel during training or primary
evaluation.

In [ ]:
ARABIC_DIACRITICS = set(
    "\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652"
)

def contains_tashkeel(text):
    return any(
        character
        in ARABIC_DIACRITICS
        for character in text
    )

contains_tashkeel(
    "وَسَيَكْتُبُونَهَا"
)

# 11. Vocabulary Construction

Vocabulary is built from the **training split only**.

In [ ]:
SPECIAL_TOKENS = [
    "<pad>",
    "<sos>",
    "<eos>",
    "<unk>",
]

PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

def build_vocab(
    sentences,
):
    lexical = sorted({
        token
        for sentence in sentences
        for token in tokenize(
            sentence
        )
    })

    itos = (
        SPECIAL_TOKENS
        + lexical
    )

    stoi = {
        token: index
        for index, token
        in enumerate(
            itos
        )
    }

    return (
        stoi,
        itos,
    )

src_stoi, src_itos = (
    build_vocab(
        train_frame[
            "source"
        ]
    )
)

tgt_stoi, tgt_itos = (
    build_vocab(
        train_frame[
            "target"
        ]
    )
)

print(
    "Source vocab:",
    len(src_itos),
)

print(
    "Target vocab:",
    len(tgt_itos),
)

# 12. OOV Analysis

In [ ]:
def oov_rate(
    sentences,
    stoi,
):
    tokens = [
        token
        for sentence in sentences
        for token in tokenize(
            sentence
        )
    ]

    if not tokens:
        return 0.0

    oov = sum(
        token
        not in stoi
        for token in tokens
    )

    return (
        oov
        / len(tokens)
    )

pd.Series({
    "dev_source_OOV": oov_rate(
        dev_frame["source"],
        src_stoi,
    ),
    "test_source_OOV": oov_rate(
        test_frame["source"],
        src_stoi,
    ),
    "dev_target_OOV": oov_rate(
        dev_frame["target"],
        tgt_stoi,
    ),
    "test_target_OOV": oov_rate(
        test_frame["target"],
        tgt_stoi,
    ),
})

# 13. Length Analysis

In [ ]:
length_frame = pd.DataFrame({
    "source_length": corpus[
        "source"
    ].str.split().str.len(),
    "target_length": corpus[
        "target"
    ].str.split().str.len(),
})

plt.figure(
    figsize=(7, 5)
)

plt.scatter(
    length_frame[
        "source_length"
    ],
    length_frame[
        "target_length"
    ],
)

plt.xlabel(
    "Source tokens"
)

plt.ylabel(
    "Target tokens"
)

plt.title(
    "Parallel Sentence Lengths"
)

plt.tight_layout()
plt.show()

# 14. Baseline System

A lexical baseline gives the neural model something meaningful to beat.

In [ ]:
lexicon = {
    "je": "i",
    "tu": "you",
    "il": "he",
    "elle": "she",
    "nous": "we",
    "vous": "you",
    "ils": "they",
    "elles": "they",
    "suis": "am",
    "es": "are",
    "est": "is",
    "sommes": "are",
    "etes": "are",
    "sont": "are",
    "ici": "here",
    "petit": "small",
    "petite": "small",
    "petits": "small",
    "petites": "small",
    "grand": "big",
    "grande": "big",
    "grands": "big",
    "grandes": "big",
}

def lexical_baseline(
    source_sentence,
):
    return " ".join(
        lexicon.get(
            token,
            token,
        )
        for token in tokenize(
            source_sentence
        )
    )

lexical_baseline(
    "elle est petite"
)

# 15. Tiny Transformer Dataset

In [ ]:
def encode_sentence(
    sentence,
    stoi,
):
    ids = [
        SOS_IDX
    ]

    ids.extend(
        stoi.get(
            token,
            UNK_IDX,
        )
        for token in tokenize(
            sentence
        )
    )

    ids.append(
        EOS_IDX
    )

    return torch.tensor(
        ids,
        dtype=torch.long,
    )

def make_batch(
    frame,
):
    source_tensors = [
        encode_sentence(
            sentence,
            src_stoi,
        )
        for sentence
        in frame["source"]
    ]

    target_tensors = [
        encode_sentence(
            sentence,
            tgt_stoi,
        )
        for sentence
        in frame["target"]
    ]

    source = pad_sequence(
        source_tensors,
        batch_first=True,
        padding_value=PAD_IDX,
    )

    target = pad_sequence(
        target_tensors,
        batch_first=True,
        padding_value=PAD_IDX,
    )

    return (
        source.to(
            DEVICE
        ),
        target.to(
            DEVICE
        ),
    )

train_source, train_target = (
    make_batch(
        train_frame
    )
)

dev_source, dev_target = (
    make_batch(
        dev_frame
    )
)

print(
    train_source.shape,
    train_target.shape,
)

# 16. Padding and Masks

Padding tokens must not contribute to:

- self-attention;
- cross-attention;
- training loss.

# 17. Positional Encoding

In [ ]:
class PositionalEncoding(
    nn.Module
):
    def __init__(
        self,
        d_model,
        max_length=64,
    ):
        super().__init__()

        positions = torch.arange(
            max_length
        ).unsqueeze(1)

        divisors = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
            )
            * (
                -math.log(
                    10000.0
                )
                / d_model
            )
        )

        encoding = torch.zeros(
            max_length,
            d_model,
        )

        encoding[
            :,
            0::2
        ] = torch.sin(
            positions
            * divisors
        )

        encoding[
            :,
            1::2
        ] = torch.cos(
            positions
            * divisors
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
        )

    def forward(
        self,
        x,
    ):
        return (
            x
            + self.encoding[
                :,
                :x.size(1),
                :
            ]
        )

# 18. Transformer Model

In [ ]:
class TranslationTransformer(
    nn.Module
):
    def __init__(
        self,
        source_vocab_size,
        target_vocab_size,
        d_model=48,
        nhead=4,
        layers=2,
        feedforward=96,
    ):
        super().__init__()

        self.d_model = (
            d_model
        )

        self.source_embedding = (
            nn.Embedding(
                source_vocab_size,
                d_model,
                padding_idx=PAD_IDX,
            )
        )

        self.target_embedding = (
            nn.Embedding(
                target_vocab_size,
                d_model,
                padding_idx=PAD_IDX,
            )
        )

        self.position = (
            PositionalEncoding(
                d_model
            )
        )

        self.transformer = (
            nn.Transformer(
                d_model=d_model,
                nhead=nhead,
                num_encoder_layers=layers,
                num_decoder_layers=layers,
                dim_feedforward=feedforward,
                dropout=0.0,
                batch_first=True,
            )
        )

        self.output = nn.Linear(
            d_model,
            target_vocab_size,
        )

    def causal_mask(
        self,
        length,
        device,
    ):
        return torch.triu(
            torch.full(
                (
                    length,
                    length,
                ),
                float("-inf"),
                device=device,
            ),
            diagonal=1,
        )

    def forward(
        self,
        source,
        decoder_input,
    ):
        source_pad = (
            source
            == PAD_IDX
        )

        target_pad = (
            decoder_input
            == PAD_IDX
        )

        target_mask = (
            self.causal_mask(
                decoder_input.size(
                    1
                ),
                decoder_input.device,
            )
        )

        source_hidden = (
            self.position(
                self.source_embedding(
                    source
                )
                * math.sqrt(
                    self.d_model
                )
            )
        )

        target_hidden = (
            self.position(
                self.target_embedding(
                    decoder_input
                )
                * math.sqrt(
                    self.d_model
                )
            )
        )

        hidden = self.transformer(
            src=source_hidden,
            tgt=target_hidden,
            tgt_mask=target_mask,
            src_key_padding_mask=source_pad,
            tgt_key_padding_mask=target_pad,
            memory_key_padding_mask=source_pad,
        )

        return self.output(
            hidden
        )

# 19. Training Objective

Use teacher forcing by shifting the target sequence:

```text
decoder input = target[:-1]
labels        = target[1:]
```

# 20. Training Loop

Validation loss determines checkpoint selection.

In [ ]:
model = TranslationTransformer(
    source_vocab_size=len(
        src_itos
    ),
    target_vocab_size=len(
        tgt_itos
    ),
).to(
    DEVICE
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.004,
)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX,
)

EPOCHS = 120

history = []

best_state = None
best_dev_loss = float(
    "inf"
)

for epoch in range(
    1,
    EPOCHS + 1,
):
    model.train()

    optimizer.zero_grad()

    train_logits = model(
        train_source,
        train_target[
            :,
            :-1
        ],
    )

    train_loss = criterion(
        train_logits.reshape(
            -1,
            len(
                tgt_itos
            ),
        ),
        train_target[
            :,
            1:
        ].reshape(-1),
    )

    train_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0,
    )

    optimizer.step()

    model.eval()

    with torch.no_grad():
        dev_logits = model(
            dev_source,
            dev_target[
                :,
                :-1
            ],
        )

        dev_loss = criterion(
            dev_logits.reshape(
                -1,
                len(
                    tgt_itos
                ),
            ),
            dev_target[
                :,
                1:
            ].reshape(-1),
        )

    train_value = float(
        train_loss.item()
    )

    dev_value = float(
        dev_loss.item()
    )

    history.append({
        "epoch": epoch,
        "train_loss": (
            train_value
        ),
        "dev_loss": (
            dev_value
        ),
    })

    if (
        dev_value
        < best_dev_loss
    ):
        best_dev_loss = (
            dev_value
        )

        best_state = copy.deepcopy(
            model.state_dict()
        )

model.load_state_dict(
    best_state
)

history_frame = pd.DataFrame(
    history
)

print(
    "Best dev loss:",
    round(
        best_dev_loss,
        4,
    ),
)

# 21. Validation Loss

In [ ]:
plt.figure(
    figsize=(7, 5)
)

plt.plot(
    history_frame[
        "epoch"
    ],
    history_frame[
        "train_loss"
    ],
    label="train",
)

plt.plot(
    history_frame[
        "epoch"
    ],
    history_frame[
        "dev_loss"
    ],
    label="dev",
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Cross-entropy loss"
)

plt.title(
    "Training and Validation Loss"
)

plt.legend()

plt.tight_layout()
plt.show()

# 22. Checkpoint Selection

The best checkpoint is selected from **development data**, never from test performance.

# 23. Greedy Decoding

In [ ]:
def greedy_translate(
    source_sentence,
    max_new_tokens=8,
):
    model.eval()

    source = encode_sentence(
        source_sentence,
        src_stoi,
    ).unsqueeze(0).to(
        DEVICE
    )

    generated = torch.tensor(
        [
            [
                SOS_IDX
            ]
        ],
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.no_grad():
        for _ in range(
            max_new_tokens
        ):
            logits = model(
                source,
                generated,
            )

            next_token = int(
                logits[
                    0,
                    -1,
                    :
                ]
                .argmax()
                .item()
            )

            generated = torch.cat(
                [
                    generated,
                    torch.tensor(
                        [
                            [
                                next_token
                            ]
                        ],
                        device=DEVICE,
                    ),
                ],
                dim=1,
            )

            if (
                next_token
                == EOS_IDX
            ):
                break

    output_tokens = []

    for token_id in generated[
        0,
        1:
    ].tolist():
        if (
            token_id
            == EOS_IDX
        ):
            break

        if token_id not in {
            PAD_IDX,
            SOS_IDX,
        }:
            output_tokens.append(
                tgt_itos[
                    token_id
                ]
            )

    return " ".join(
        output_tokens
    )

# 24. Test Translation

Only after model selection do we evaluate the held-out test set.

In [ ]:
test_results = []

for row in test_frame.itertuples(
    index=False
):
    baseline = (
        lexical_baseline(
            row.source
        )
    )

    transformer = (
        greedy_translate(
            row.source
        )
    )

    test_results.append({
        "source": row.source,
        "reference": row.target,
        "baseline": baseline,
        "transformer": (
            transformer
        ),
    })

test_results = pd.DataFrame(
    test_results
)

test_results

# 25. BLEU-Style Metric

This small educational implementation uses unigram/bigram modified precision plus a
brevity penalty.

In [ ]:
def ngrams(
    tokens,
    n,
):
    return [
        tuple(
            tokens[
                i:
                i+n
            ]
        )
        for i in range(
            len(tokens)
            - n
            + 1
        )
    ]

def corpus_bleu2(
    references,
    hypotheses,
):
    clipped_totals = [
        0.0,
        0.0,
    ]

    candidate_totals = [
        0.0,
        0.0,
    ]

    ref_length = 0
    hyp_length = 0

    for reference, hypothesis in zip(
        references,
        hypotheses,
    ):
        ref_tokens = tokenize(
            reference
        )

        hyp_tokens = tokenize(
            hypothesis
        )

        ref_length += len(
            ref_tokens
        )

        hyp_length += len(
            hyp_tokens
        )

        for order in [
            1,
            2,
        ]:
            ref_counts = Counter(
                ngrams(
                    ref_tokens,
                    order,
                )
            )

            hyp_counts = Counter(
                ngrams(
                    hyp_tokens,
                    order,
                )
            )

            clipped_totals[
                order - 1
            ] += sum(
                min(
                    count,
                    ref_counts[
                        gram
                    ],
                )
                for gram, count
                in hyp_counts.items()
            )

            candidate_totals[
                order - 1
            ] += sum(
                hyp_counts.values()
            )

    precisions = []

    for index in range(2):
        numerator = (
            clipped_totals[
                index
            ]
            + 1.0
        )

        denominator = (
            candidate_totals[
                index
            ]
            + 1.0
        )

        precisions.append(
            numerator
            / denominator
        )

    if hyp_length == 0:
        return 0.0

    if hyp_length > ref_length:
        bp = 1.0
    else:
        bp = math.exp(
            1.0
            - ref_length
            / hyp_length
        )

    score = bp * math.exp(
        np.mean(
            [
                math.log(
                    p
                )
                for p in precisions
            ]
        )
    )

    return (
        100.0
        * score
    )

# 26. chrF-Style Metric

In [ ]:
def character_f1(
    reference,
    hypothesis,
):
    ref_chars = [
        char
        for char in reference
        if not char.isspace()
    ]

    hyp_chars = [
        char
        for char in hypothesis
        if not char.isspace()
    ]

    if (
        not ref_chars
        or not hyp_chars
    ):
        return 0.0

    overlap = sum(
        (
            Counter(
                ref_chars
            )
            & Counter(
                hyp_chars
            )
        ).values()
    )

    precision = (
        overlap
        / len(
            hyp_chars
        )
    )

    recall = (
        overlap
        / len(
            ref_chars
        )
    )

    if (
        precision
        + recall
        == 0
    ):
        return 0.0

    return (
        100.0
        * 2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )

# 27. Exact Match

In [ ]:
def exact_match(
    reference,
    hypothesis,
):
    return float(
        reference
        == hypothesis
    )

# 28. Sentence-Level Scores

In [ ]:
score_rows = []

for row in test_results.itertuples(
    index=False
):
    for system_name in [
        "baseline",
        "transformer",
    ]:
        hypothesis = getattr(
            row,
            system_name,
        )

        score_rows.append({
            "source": row.source,
            "reference": (
                row.reference
            ),
            "system": (
                system_name
            ),
            "hypothesis": (
                hypothesis
            ),
            "char_f1": (
                character_f1(
                    row.reference,
                    hypothesis,
                )
            ),
            "exact": (
                exact_match(
                    row.reference,
                    hypothesis,
                )
            ),
        })

sentence_scores = pd.DataFrame(
    score_rows
)

sentence_scores

# 29. Mean and Standard Deviation

In [ ]:
sentence_summary = (
    sentence_scores
    .groupby(
        "system"
    )[
        [
            "char_f1",
            "exact",
        ]
    ]
    .agg(
        [
            "mean",
            "std",
        ]
    )
)

sentence_summary

# 30. Bootstrap 95% CI

In [ ]:
def bootstrap_mean_ci(
    values,
    n_bootstrap=3000,
    seed=42,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    rng = np.random.default_rng(
        seed
    )

    means = np.empty(
        n_bootstrap
    )

    for i in range(
        n_bootstrap
    ):
        sample = rng.choice(
            values,
            size=len(
                values
            ),
            replace=True,
        )

        means[i] = (
            sample.mean()
        )

    return (
        float(
            np.quantile(
                means,
                0.025,
            )
        ),
        float(
            np.quantile(
                means,
                0.975,
            )
        ),
    )

ci_rows = []

for system_name in [
    "baseline",
    "transformer",
]:
    values = sentence_scores[
        sentence_scores[
            "system"
        ]
        == system_name
    ][
        "char_f1"
    ].to_numpy()

    lower, upper = (
        bootstrap_mean_ci(
            values,
            seed=SEED,
        )
    )

    ci_rows.append({
        "system": system_name,
        "mean_char_f1": (
            values.mean()
        ),
        "CI95_lower": lower,
        "CI95_upper": upper,
    })

ci_frame = pd.DataFrame(
    ci_rows
)

ci_frame

# 31. Paired System Comparison

Systems translate the same sentences, so uncertainty analysis should preserve pairing.

In [ ]:
def paired_bootstrap_difference(
    scores_a,
    scores_b,
    n_bootstrap=5000,
    seed=42,
):
    scores_a = np.asarray(
        scores_a,
        dtype=float,
    )

    scores_b = np.asarray(
        scores_b,
        dtype=float,
    )

    rng = np.random.default_rng(
        seed
    )

    n = len(
        scores_a
    )

    differences = np.empty(
        n_bootstrap
    )

    for i in range(
        n_bootstrap
    ):
        indices = rng.integers(
            0,
            n,
            size=n,
        )

        differences[i] = (
            scores_b[
                indices
            ].mean()
            - scores_a[
                indices
            ].mean()
        )

    return {
        "observed_difference": float(
            scores_b.mean()
            - scores_a.mean()
        ),
        "CI95_lower": float(
            np.quantile(
                differences,
                0.025,
            )
        ),
        "CI95_upper": float(
            np.quantile(
                differences,
                0.975,
            )
        ),
        "P_B_better": float(
            np.mean(
                differences
                > 0
            )
        ),
    }

baseline_scores = sentence_scores[
    sentence_scores[
        "system"
    ]
    == "baseline"
][
    "char_f1"
].to_numpy()

transformer_scores = sentence_scores[
    sentence_scores[
        "system"
    ]
    == "transformer"
][
    "char_f1"
].to_numpy()

paired_bootstrap_difference(
    baseline_scores,
    transformer_scores,
)

# 32. Approximate Randomization

In [ ]:
def approximate_randomization(
    scores_a,
    scores_b,
    iterations=10000,
    seed=42,
):
    scores_a = np.asarray(
        scores_a,
        dtype=float,
    )

    scores_b = np.asarray(
        scores_b,
        dtype=float,
    )

    observed = abs(
        scores_b.mean()
        - scores_a.mean()
    )

    rng = np.random.default_rng(
        seed
    )

    extreme = 0

    for _ in range(
        iterations
    ):
        swap = rng.random(
            len(
                scores_a
            )
        ) < 0.5

        perm_a = np.where(
            swap,
            scores_b,
            scores_a,
        )

        perm_b = np.where(
            swap,
            scores_a,
            scores_b,
        )

        difference = abs(
            perm_b.mean()
            - perm_a.mean()
        )

        if (
            difference
            >= observed
        ):
            extreme += 1

    return (
        extreme + 1
    ) / (
        iterations + 1
    )

approximate_randomization(
    baseline_scores,
    transformer_scores,
)

# 33. Error Taxonomy

In [ ]:
error_taxonomy = pd.DataFrame(
    [
        ("Lexical", "wrong translated content word"),
        ("Morphology", "wrong inflection"),
        ("Agreement", "person/number/gender mismatch"),
        ("Omission", "source information missing"),
        ("Addition", "unsupported content introduced"),
        ("Word order", "target order incorrect"),
        ("Unknown token", "unseen source/target unit"),
        ("Diacritization", "tashkeel missing or incorrect"),
    ],
    columns=[
        "Error type",
        "Description",
    ],
)

error_taxonomy

# 34. Automatic Error Flags

Automatic flags are only diagnostics. They do not replace manual linguistic analysis.

In [ ]:
def automatic_error_flags(
    reference,
    hypothesis,
):
    reference_tokens = tokenize(
        reference
    )

    hypothesis_tokens = tokenize(
        hypothesis
    )

    return {
        "length_mismatch": (
            len(
                reference_tokens
            )
            != len(
                hypothesis_tokens
            )
        ),
        "unknown_token": (
            "<unk>"
            in hypothesis_tokens
        ),
        "exact_mismatch": (
            reference
            != hypothesis
        ),
    }

automatic_error_flags(
    "she is small",
    "she small",
)

# 35. Manual Error Sheet

In [ ]:
manual_error_sheet = (
    test_results[
        [
            "source",
            "reference",
            "transformer",
        ]
    ]
    .copy()
)

manual_error_sheet[
    "error_category"
] = ""

manual_error_sheet[
    "notes"
] = ""

manual_error_sheet

# 36. Direction-Specific Analysis

Do not assume Arabic→English and English→Arabic behave symmetrically.

Each direction should receive its own:

- training run;
- decoding settings;
- metric table;
- error analysis.

# 37. Arabic-Specific Analysis

For Arabic, inspect:

- clitic errors;
- gender and number agreement;
- attached pronouns;
- named entities;
- diacritization;
- lexical ambiguity.

# 38. Tashkeel Preservation

In [ ]:
def tashkeel_sequence(
    text,
):
    return "".join(
        character
        for character in text
        if character
        in ARABIC_DIACRITICS
    )

example_reference = (
    "كِتَابُهُمَا"
)

example_prediction = (
    "كتابُهما"
)

pd.Series({
    "reference_tashkeel": (
        tashkeel_sequence(
            example_reference
        )
    ),
    "prediction_tashkeel": (
        tashkeel_sequence(
            example_prediction
        )
    ),
    "surface_exact": (
        example_reference
        == example_prediction
    ),
})

# 39. Experiment Configuration

In [ ]:
experiment_config = pd.Series(
    {
        "direction": (
            "French→English"
        ),
        "seed": SEED,
        "train_pairs": len(
            train_frame
        ),
        "dev_pairs": len(
            dev_frame
        ),
        "test_pairs": len(
            test_frame
        ),
        "tokenization": (
            "whitespace"
        ),
        "model": (
            "tiny Transformer"
        ),
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "learning_rate": (
            0.004
        ),
        "decoding": (
            "greedy"
        ),
        "checkpoint_selection": (
            "lowest dev loss"
        ),
    },
    name="Experiment configuration",
)

experiment_config

# 40. Results Table

In [ ]:
references = (
    test_results[
        "reference"
    ].tolist()
)

baseline_hypotheses = (
    test_results[
        "baseline"
    ].tolist()
)

transformer_hypotheses = (
    test_results[
        "transformer"
    ].tolist()
)

result_rows = []

for system_name, hypotheses in [
    (
        "Lexical baseline",
        baseline_hypotheses,
    ),
    (
        "Tiny Transformer",
        transformer_hypotheses,
    ),
]:
    char_scores = np.array(
        [
            character_f1(
                reference,
                hypothesis,
            )
            for reference, hypothesis
            in zip(
                references,
                hypotheses,
            )
        ]
    )

    lower, upper = (
        bootstrap_mean_ci(
            char_scores,
            seed=SEED,
        )
    )

    result_rows.append({
        "System": system_name,
        "BLEU-2": corpus_bleu2(
            references,
            hypotheses,
        ),
        "Character F1 mean": float(
            char_scores.mean()
        ),
        "Character F1 SD": float(
            char_scores.std(
                ddof=1
            )
        )
        if len(
            char_scores
        ) > 1
        else 0.0,
        "Character F1 95% CI": (
            f"[{lower:.2f}, "
            f"{upper:.2f}]"
        ),
        "Exact accuracy": float(
            np.mean(
                [
                    exact_match(
                        reference,
                        hypothesis,
                    )
                    for reference, hypothesis
                    in zip(
                        references,
                        hypotheses,
                    )
                ]
            )
        ),
    })

results_table = pd.DataFrame(
    result_rows
)

results_table

# 41. Ablation Design

Ablations isolate which design choice causes an improvement.

Possible MT ablations:

- word vs subword tokenization;
- vocalized vs unvocalized Arabic;
- greedy vs beam decoding;
- pretrained vs randomly initialized model;
- full fine-tuning vs PEFT;
- monolingual vs multilingual checkpoint.

# 42. Baseline vs Transformer

A meaningful conclusion should compare:

- corpus metric difference;
- sentence-level mean difference;
- uncertainty;
- significance;
- failure categories.

Never conclude that a model is superior from one headline metric alone.

# 43. Pretrained Fine-Tuning Workflow

A modern practical experiment commonly follows:

```text
parallel data
    ↓
tokenizer
    ↓
pretrained encoder-decoder model
    ↓
Seq2Seq training
    ↓
dev-set model selection
    ↓
generate test translations
    ↓
BLEU / chrF / COMET / semantic metrics
    ↓
confidence intervals + significance
    ↓
error analysis
```

# 44. Current Hugging Face Seq2Seq Template

This optional template is disabled because it requires an external checkpoint and
dataset objects.

The current Transformers translation workflow uses:

- `AutoTokenizer`;
- `text_target=...` for target tokenization;
- `AutoModelForSeq2SeqLM`;
- `Seq2SeqTrainingArguments`;
- `Seq2SeqTrainer`;
- `processing_class=tokenizer`;
- `predict_with_generate=True`.

In [ ]:
RUN_HUGGING_FACE_FINE_TUNING = False

hugging_face_template = '''
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

checkpoint = "your-seq2seq-checkpoint"

tokenizer = AutoTokenizer.from_pretrained(
    checkpoint
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint
)

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["source"],
        max_length=128,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=128,
        truncation=True,
    )

    model_inputs["labels"] = labels[
        "input_ids"
    ]

    return model_inputs

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="mt_experiment",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    predict_with_generate=True,
    save_total_limit=2,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
'''

print(
    hugging_face_template
)

# 45. Evaluation During Fine-Tuning

Development metrics can be computed during training.

The final test set should remain reserved for the selected checkpoint.

# 46. Checkpointing

Save:

- model checkpoint;
- tokenizer;
- training arguments;
- seed;
- evaluation script;
- package versions.

A metric value without its associated configuration is difficult to reproduce.

# 47. Test-Set Discipline

Do not repeatedly modify the model after inspecting test performance.

Repeated test-driven tuning gradually turns the test set into a development set.

# 48. Final Report Structure

A concise MT experiment report should contain:

1. research question;
2. dataset;
3. preprocessing;
4. model;
5. training settings;
6. decoding;
7. metrics;
8. confidence intervals;
9. significance;
10. error analysis;
11. limitations;
12. reproducibility information.

# 49. Limitations

The executable model in this notebook is intentionally tiny.

It is useful for learning the experimental workflow, not for estimating the real-world
quality of production MT models.

# 50. Reproducibility Checklist

Before publishing or sharing an MT experiment, confirm that you recorded:

- dataset source and version;
- train/dev/test split;
- preprocessing;
- tokenizer;
- model checkpoint;
- random seed;
- optimizer and learning rate;
- epoch count;
- checkpoint-selection criterion;
- decoding settings;
- BLEU/chrF implementation;
- COMET checkpoint if used;
- confidence-interval method;
- significance test;
- hardware;
- package versions.

# 51. Knowledge Check

1. Why split data before vocabulary construction?
2. What is data leakage?
3. Why should test data not determine checkpoint selection?
4. Why include a baseline?
5. What does validation loss control?
6. Why are sentence-level scores useful?
7. What does a bootstrap CI quantify?
8. Why should system comparison be paired?
9. Why use both metrics and error analysis?
10. What does an ablation study test?
11. Why are MT directions evaluated separately?
12. Why should vocalized Arabic preserve tashkeel?
13. What does `text_target` represent in Seq2Seq tokenization?
14. Why use `predict_with_generate=True` for generation-based evaluation?
15. What must be included in a reproducibility checklist?

# 52. Project Exercises

## Exercise 1
Replace the toy corpus with a larger parallel dataset.

## Exercise 2
Add a SentencePiece or BPE tokenizer.

## Exercise 3
Train the reverse translation direction.

## Exercise 4
Implement beam search.

## Exercise 5
Evaluate with SacreBLEU BLEU and chrF++.

## Exercise 6
Add COMET.

## Exercise 7
Add multilingual SentenceTransformer similarity.

## Exercise 8
Compare word, subword, and character tokenization.

## Exercise 9
Run Arabic→English and English→Arabic as separate experiments.

## Exercise 10
Create a final results table containing BLEU, chrF, COMET, semantic similarity, mean,
standard deviation, 95% confidence interval, p-value, and effect size.

## Challenge Project

Run the same train/dev/test corpus through multiple model families:

- MarianMT;
- M2M-100;
- NLLB;
- mT5;
- a Transformer trained from scratch.

Then compare:

- zero-shot;
- fine-tuned;
- Arabic→English;
- English→Arabic;
- word/subword/character settings where applicable;
- BLEU;
- chrF;
- COMET;
- semantic similarity;
- confidence intervals;
- significance.

This produces a research-grade comparative MT experiment.

# 53. Module 9 Summary

Module 9 progressed through:

- Statistical Machine Translation;
- encoder-decoder Neural Machine Translation;
- attention;
- Transformer MT;
- MarianMT, M2M-100, and NLLB;
- Arabic↔English MT;
- morphology, tokenization, and tashkeel;
- BLEU, chrF, COMET, semantic evaluation;
- statistical significance;
- end-to-end experiment design.

## Next Module

**Module 10 — Advanced Applications**

The next lesson is:

**Lesson 57: Advanced Information Retrieval — Neural Retrieval, Re-Ranking, and
Production Search Pipelines**

# References

- Vaswani, A. et al. *Attention Is All You Need*.
- Sutskever, I. et al. *Sequence to Sequence Learning with Neural Networks*.
- Bahdanau, D. et al. *Neural Machine Translation by Jointly Learning to Align and Translate*.
- Papineni, K. et al. *BLEU: a Method for Automatic Evaluation of Machine Translation*.
- Popović, M. work on chrF.
- Rei, R. et al. work on COMET.
- Hugging Face Transformers translation-task documentation.